In [19]:
import os

def folder_has_any_file(folder_path):
    return any(
        os.path.isfile(os.path.join(folder_path, f)) and not f.startswith(".")
        for f in os.listdir(folder_path)
    )

folder_path = "../../docs"
if folder_has_any_file("../../docs"):
    print("yes")
else:
    print("no")

no


In [4]:
import os
import sys
from langchain_openai import ChatOpenAI
sys.path.append('../..')
from langchain_mongodb import MongoDBAtlasVectorSearch
from pymongo import MongoClient
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

from service.chatbot.langchain.embedding_langchain import embedding
from service.chatbot.langchain.generating_langchain import make_generate
from service.chatbot.langchain.retrieving_langchain import make_retrieve

from langchain_openai import OpenAIEmbeddings  


In [6]:
client = MongoClient(os.environ["MONGODB_ATLAS_CLUSTER_URI"])

db = os.environ["DB_NAME"]
collection_name = os.environ["COLLECTION_NAME"]
search_index = os.environ["ATLAS_VECTOR_SEARCH_INDEX_NAME"]

collection = client[db][collection_name]


llm = ChatOpenAI(
        base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
        model=os.environ["AZURE_OPENAI_COMP_DEPLOYMENT_NAME"],
        api_key=os.environ["AZURE_OPENAI_API_VERSION"]
    )


embedder = OpenAIEmbeddings(
    model="azure-text-embedding-3-large",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ['AZURE_OPENAI_ENDPOINT']
)
vectorstore_db = MongoDBAtlasVectorSearch(
            collection=collection,
            embedding=embedder,  # same embedding model used when saving
            index_name=search_index,
            relevance_score_fn="cosine"
        )




In [8]:
from langgraph.graph import MessagesState, StateGraph
graph_builder = StateGraph(MessagesState)

In [9]:
from langchain_core.tools import tool


@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """Retrieve information related to a query."""
    retrieved_docs = vectorstore_db.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n" f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [10]:
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode


# Step 1: Generate an AIMessage that may include a tool-call to be sent.
def query_or_respond(state: MessagesState):
    """Generate tool call for retrieval or respond."""
    llm_with_tools = llm.bind_tools([retrieve])
    response = llm_with_tools.invoke(state["messages"])
    # MessagesState appends messages to state instead of overwriting
    return {"messages": [response]}


# Step 2: Execute the retrieval.
tools = ToolNode([retrieve])


# Step 3: Generate a response using the retrieved content.
def generate(state: MessagesState):
    """Generate answer."""
    # Get generated ToolMessages
    recent_tool_messages = []
    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]

    # Format into prompt
    docs_content = "\n\n".join(doc.content for doc in tool_messages)
    system_message_content = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, say that you "
        "don't know. Use three sentences maximum and keep the "
        "answer concise."
        "\n\n"
        f"{docs_content}"
    )
    conversation_messages = [
        message
        for message in state["messages"]
        if message.type in ("human", "system")
        or (message.type == "ai" and not message.tool_calls)
    ]
    prompt = [SystemMessage(system_message_content)] + conversation_messages

    # Run
    response = llm.invoke(prompt)
    return {"messages": [response]}

In [11]:
from langgraph.graph import END
from langgraph.prebuilt import ToolNode, tools_condition

graph_builder.add_node(query_or_respond)
graph_builder.add_node(tools)
graph_builder.add_node(generate)

graph_builder.set_entry_point("query_or_respond")
graph_builder.add_conditional_edges(
    "query_or_respond",
    tools_condition,
    {END: END, "tools": "tools"},
)
graph_builder.add_edge("tools", "generate")
graph_builder.add_edge("generate", END)

graph = graph_builder.compile()

In [12]:
input_message = "Summarize lab 3"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()



================================ Human Message =================================

Summarize lab 3
================================== Ai Message ==================================
Tool Calls:
  retrieve (call_gdRMHJBYBPriOEvKXifkQ9ph)
 Call ID: call_gdRMHJBYBPriOEvKXifkQ9ph
  Args:
    query: lab 3 summary
================================= Tool Message =================================
Name: retrieve

Source: {'_id': '0', 'producer': 'pdfTeX-1.40.19', 'creator': 'LaTeX with hyperref package', 'creationdate': '2025-02-04T14:24:49-05:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2025-02-04T14:24:49-05:00', 'trapped': '/False', 'ptex.fullbanner': 'This is MiKTeX-pdfTeX 2.9.6642 (1.40.19)', 'source': '../../docs/Lab03_315.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'start_index': 0}
Content: CIT315 Lab3
Lab3 - CIT315
Goals
The primary goal of this project is to write, test and complete a c program which uses functions
and recursion. It is critical that

In [26]:
import asyncio
from file_handling_service import FileHandlingService
from langchain_openai import OpenAIEmbeddings  
from pymongo import MongoClient
from langchain_openai import ChatOpenAI
import os


client = MongoClient(os.environ["MONGODB_ATLAS_CLUSTER_URI"])
db = os.environ["DB_NAME"]
collection_name = os.environ["COLLECTION_NAME"]
search_index = os.environ["ATLAS_VECTOR_SEARCH_INDEX_NAME"]
collection = client[db][collection_name]

embedder = OpenAIEmbeddings(
        model="azure-text-embedding-3-large",
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=os.environ['AZURE_OPENAI_ENDPOINT']
    )

llm = ChatOpenAI(
            base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
            model=os.environ["AZURE_OPENAI_COMP_DEPLOYMENT_NAME"],
            api_key=os.environ["AZURE_OPENAI_API_VERSION"]
        )

self_service = FileHandlingService(llm, "../docs", embedder, collection, search_index)
print(self_service.folder_has_any_file())
if self_service.folder_has_any_file():
        for file in os.listdir(self_service.pdf_path):
            new_file_path = os.path.join(self_service.pdf_path, file)
            if os.path.isfile(new_file_path):
                docs = await self_service.loading()
                all_splits = self_service.splitting(docs)
                vectorstore_db = self_service.embed_documents(all_splits, embedder, collection, search_index)
                os.remove(new_file_path)
                print("reached")


False
